In [118]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [119]:
DIR_PATH = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "outputs/bcred-20260331.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)

DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [120]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 1293 entries, 0 to 1292
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   investments       1293 non-null   str    
 1   reference_rate    1293 non-null   str    
 2   interest_rate     1205 non-null   str    
 3   industry          1293 non-null   str    
 4   instrument        1293 non-null   str    
 5   investment_type   1293 non-null   str    
 6   acquisition_date  1289 non-null   str    
 7   maturity_date     1201 non-null   str    
 8   par_amount_units  1287 non-null   str    
 9   net_assets        1291 non-null   float64
 10  cost              1291 non-null   float64
 11  fair_value        1291 non-null   float64
 12  part              1293 non-null   str    
 13  table_index       1293 non-null   int64  
 14  row_index         1293 non-null   int64  
dtypes: float64(3), int64(2), str(10)
memory usage: 151.7 KB


In [121]:
# df_investment[df_investment["investments"].str.contains("Platte River", na=False)]

In [122]:
df_investment.head(275)

,investments,reference_rate,interest_rate,industry,instrument,investment_type,acquisition_date,maturity_date,par_amount_units,net_assets,cost,fair_value,part,table_index,row_index
0,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.22%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,41906,0.02,29763.0,8882.0,bcred-20260331,0,7
1,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,6096,0.00,4373.0,1292.0,bcred-20260331,0,8
2,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,37217,0.02,24850.0,6792.0,bcred-20260331,0,9
3,Atlas CC Acquisition Corp.,SOFR + 4.00 %,7.93%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,18518,0.04,13604.0,18193.0,bcred-20260331,0,10
4,"Corfin Holdings, Inc.",SOFR + 5.25 %,9.02%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,1/7/2021,12/27/2027,32050,0.07,32034.0,32050.0,bcred-20260331,0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,"Spectrum Safety Solutions Purchaser, LLC",E + 4.50 %,6.63%,"electronic equipment, instruments & components",first lien debt,first lien debt - non-controlled/non-affiliated,7/1/2024,7/1/2031,63476,0.16,67411.0,73369.0,bcred-20260331,10,24
271,"Spectrum Safety Solutions Purchaser, LLC",SOFR + 4.50 %,8.20%,"electronic equipment, instruments & components",first lien debt,first lien debt - non-controlled/non-affiliated,7/1/2024,7/1/2031,297343,0.66,293294.0,297343.0,bcred-20260331,10,25
272,"LPW Group Holdings, Inc.",SOFR + 6.00 %,9.77%,energy equipment & services,first lien debt,first lien debt - non-controlled/non-affiliated,3/15/2024,3/15/2031,32258,0.07,31579.0,32258.0,bcred-20260331,11,6
273,"EOC Borrower, LLC",SOFR + 2.75 %,6.42%,entertainment,first lien debt,first lien debt - non-controlled/non-affiliated,2/3/2026,3/24/2032,3990,0.01,3985.0,3986.0,bcred-20260331,11,8


In [123]:
# convert cost 
for col in ["cost", "fair_value"]:
    df_investment[col] = (
        df_investment[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .replace("nan", pd.NA)
    )

    df_investment[col] = pd.to_numeric(df_investment[col], errors="coerce").astype("Int64")

df_investment.drop(columns=["part", "table_index", "row_index"], inplace=True)
df_investment

,investments,reference_rate,interest_rate,industry,instrument,investment_type,acquisition_date,maturity_date,par_amount_units,net_assets,cost,fair_value
0,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.22%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,41906,0.02,29763,8882
1,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,6096,0.00,4373,1292
2,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,37217,0.02,24850,6792
3,Atlas CC Acquisition Corp.,SOFR + 4.00 %,7.93%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,18518,0.04,13604,18193
4,"Corfin Holdings, Inc.",SOFR + 5.25 %,9.02%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,1/7/2021,12/27/2027,32050,0.07,32034,32050
...,...,...,...,...,...,...,...,...,...,...,...,...
1288,BCRED Verdelite JV LP - LP Interest,,NaN,specialty retail,equity and other,investments in joint ventures,10/21/2022,NaN,NaN,0.22,117706,97149
1289,Cash and Cash Equivalents,Cash and Cash Equivalents Cash and Cash Equiva...,Cash and Cash Equivalents,specialty retail,equity and other,cash and cash equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,NaN,<NA>,<NA>
1290,Fidelity Investments Money Market Treasury Por...,,3.55%,specialty retail,equity and other,cash and cash equivalents,NaN,NaN,NaN,0.01,6563,6563
1291,State Street Institutional U.S. Government Mon...,,3.52%,specialty retail,equity and other,cash and cash equivalents,NaN,NaN,NaN,0.02,10440,10440


In [124]:
df_investment["profit"] = df_investment["fair_value"] - df_investment["cost"]
df_investment["is_profit"] = df_investment["profit"] > 0
df_investment["is_loss"] = df_investment["profit"] < 0
df_investment["is_breakeven"] = df_investment["profit"] == 0
df_investment

,investments,reference_rate,interest_rate,industry,instrument,investment_type,acquisition_date,maturity_date,par_amount_units,net_assets,cost,fair_value,profit,is_profit,is_loss,is_breakeven
0,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.22%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,41906,0.02,29763,8882,-20881,False,True,False
1,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,6096,0.00,4373,1292,-3081,False,True,False
2,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,37217,0.02,24850,6792,-18058,False,True,False
3,Atlas CC Acquisition Corp.,SOFR + 4.00 %,7.93%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,7/25/2025,5/25/2029,18518,0.04,13604,18193,4589,True,False,False
4,"Corfin Holdings, Inc.",SOFR + 5.25 %,9.02%,aerospace & defense,first lien debt,first lien debt - non-controlled/non-affiliated,1/7/2021,12/27/2027,32050,0.07,32034,32050,16,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1288,BCRED Verdelite JV LP - LP Interest,,NaN,specialty retail,equity and other,investments in joint ventures,10/21/2022,NaN,NaN,0.22,117706,97149,-20557,False,True,False
1289,Cash and Cash Equivalents,Cash and Cash Equivalents Cash and Cash Equiva...,Cash and Cash Equivalents,specialty retail,equity and other,cash and cash equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1290,Fidelity Investments Money Market Treasury Por...,,3.55%,specialty retail,equity and other,cash and cash equivalents,NaN,NaN,NaN,0.01,6563,6563,0,False,False,True
1291,State Street Institutional U.S. Government Mon...,,3.52%,specialty retail,equity and other,cash and cash equivalents,NaN,NaN,NaN,0.02,10440,10440,0,False,False,True


In [125]:
df_summary = pd.DataFrame({
    "a": [
        "Total Investment Count",
        "Total Cost", 
        "Total FV", 
        "P&L"
    ],
    "b": [
        len(df_investment),
        int(df_investment["cost"].sum()),
        int(df_investment["fair_value"].sum()),
        int(df_investment["fair_value"].sum())-int(df_investment["cost"].sum()),
    ]
})

df_summary

,a,b
0,Total Investment Count,1293
1,Total Cost,83006982
2,Total FV,81123110
3,P&L,-1883872


In [126]:
# investments-Level Analysis
df_investments_level_analysis = (
    df_investment.groupby("investments")
      .agg(
          total_investments=("investments", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_investments_level_analysis.reset_index(names="investments", inplace=True)
df_investments_level_analysis["avg_profit"] = round(df_investments_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_investments_level_analysis["win_rate"] = round(
    df_investments_level_analysis["profitable_count"] /
    df_investments_level_analysis["total_investments"]
, 2)

df_investments_level_analysis["breakeven_rate"] = round(
    df_investments_level_analysis["breakeven_count"] /
    df_investments_level_analysis["total_investments"]
, 2)

df_investments_level_analysis["loss_rate"] = round(
    df_investments_level_analysis["loss_count"] /
    df_investments_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_investments_level_analysis = df_investments_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "investments": "Investments",
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Investments",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_investments_level_analysis = df_investments_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_investments_level_analysis)

,Investments,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
259,"Dropbox, Inc.",2,1603086,1599228,-3858,-1929.0,-1358,-2500,0,0.0,0,0.0,2,1.0
420,"IRI Group Holdings, Inc.",1,1571509,1588160,16651,16651.0,16651,16651,1,1.0,0,0.0,0,0.0
81,BCRED Emerald JV LP - LP Interest,1,1836000,1539677,-296323,-296323.0,-296323,-296323,0,0.0,0,0.0,1,1.0
387,"Guidehouse, Inc.",1,1245071,1251898,6827,6827.0,6827,6827,1,1.0,0,0.0,0,0.0
433,"Inovalon Holdings, Inc.",2,1257670,1207576,-50094,-25047.0,-14949,-35145,0,0.0,0,0.0,2,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,"Material+ Holding Company, LLC - Class A Prefe...",1,0,0,0,0.0,0,0,0,0.0,1,1.0,0,0.0
147,"CFCo, LLC (Benefytt Technologies, Inc.) - Clas...",1,0,0,0,0.0,0,0,0,0.0,1,1.0,0,0.0
146,"CFCo, LLC (Benefytt Technologies, Inc.)",1,12571,0,-12571,-12571.0,-12571,-12571,0,0.0,0,0.0,1,1.0
126,"Box Co-Invest Blocker, LLC - (BP Alpha Holding...",1,390,0,-390,-390.0,-390,-390,0,0.0,0,0.0,1,1.0


In [127]:
# investments_type-Level Analysis
df_investmentsType_level_analysis = (
    df_investment.groupby("investment_type")
      .agg(
          total_investments=("investments", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_investmentsType_level_analysis.reset_index(names="investment_type", inplace=True)
df_investmentsType_level_analysis["avg_profit"] = round(df_investmentsType_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_investmentsType_level_analysis["win_rate"] = round(
    df_investmentsType_level_analysis["profitable_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["breakeven_rate"] = round(
    df_investmentsType_level_analysis["breakeven_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["loss_rate"] = round(
    df_investmentsType_level_analysis["loss_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_investmentsType_level_analysis = df_investmentsType_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "investment_type":"Investment Type",
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Investment Type",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_investmentsType_level_analysis = df_investmentsType_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_investmentsType_level_analysis)

,Investment Type,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
6,first lien debt - non-controlled/non-affiliated,972,73626433,72447160,-1179273,-1213.24,44246,-329098,509,0.52,19,0.02,444,0.46
8,second lien debt - non-controlled/non-affiliated,46,3283897,3051931,-231966,-5042.74,2907,-41046,11,0.24,1,0.02,34,0.74
7,investments in joint ventures,2,1953706,1636826,-316880,-158440.0,-20557,-296323,0,0.0,0,0.0,2,1.0
1,equity and other - controlled/affiliated (excl...,12,1210988,1243135,32147,2922.45,12834,0,4,0.33,7,0.58,0,0.0
3,equity and other - non-controlled/non-affiliated,72,689503,749088,59585,827.57,30653,-45455,41,0.57,9,0.12,22,0.31
0,cash and cash equivalents,4,653729,653729,0,0.0,0,0,0,0.0,3,0.75,0,0.0
4,first lien debt - controlled/affiliated,10,697770,551473,-146297,-14629.7,64,-57075,1,0.1,1,0.1,8,0.8
9,structured finance obligations - debt instrume...,98,437449,412348,-25101,-256.13,53,-1830,3,0.03,0,0.0,95,0.97
10,structured finance obligations - equity instru...,47,329028,256137,-72891,-1550.87,1163,-10958,8,0.17,1,0.02,38,0.81
11,unsecured debt - non-controlled/non-affiliated,24,82894,80108,-2786,-116.08,72,-2728,4,0.17,0,0.0,20,0.83


In [128]:
# Industry-Level Analysis
df_industry_level_analysis = (
    df_investment.groupby("industry")
      .agg(
          total_investments=("investments", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_industry_level_analysis.reset_index(names="industry", inplace=True)
df_industry_level_analysis["avg_profit"] = round(df_industry_level_analysis["avg_profit"], 2)


df_industry_level_analysis["win_rate"] = round(
    df_industry_level_analysis["profitable_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["breakeven_rate"] = round(
    df_industry_level_analysis["breakeven_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["loss_rate"] = round(
    df_industry_level_analysis["loss_count"] /
    df_industry_level_analysis["total_investments"]
, 2)


df_industry_level_analysis = df_industry_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %",
    "industry": "Industry"
}
desired_order = [
    "Industry",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_industry_level_analysis = df_industry_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_industry_level_analysis)

,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
42,software,221,22328547,21560513,-768034,-3475.27,44246,-329098,104,0.47,4,0.02,113,0.51
39,professional services,106,8956217,8937116,-19101,-180.2,13127,-57075,64,0.6,5,0.05,37,0.35
7,commercial services & supplies,89,6585998,6597267,11269,126.62,4507,-5967,59,0.66,3,0.03,27,0.3
23,health care providers & services,94,6910543,6447857,-462686,-4922.19,9106,-109033,28,0.3,7,0.07,59,0.63
28,insurance,75,4872323,4860613,-11710,-156.13,12047,-41038,49,0.65,6,0.08,20,0.27
24,health care technology,47,4426905,4446500,19595,416.91,30653,-10217,19,0.4,1,0.02,27,0.57
30,it services,43,3868043,3768147,-99896,-2323.16,17700,-35145,18,0.42,0,0.0,25,0.58
19,financial services,164,2496711,2428757,-67954,-414.35,9816,-10958,25,0.15,1,0.01,138,0.84
43,specialty retail,13,2686800,2364850,-321950,-26829.17,241,-296323,3,0.23,3,0.23,6,0.46
11,diversified consumer services,39,2316913,2286436,-30477,-781.46,8756,-24758,15,0.38,0,0.0,24,0.62


In [129]:
# asset_list = list(df_asset_class_level_analysis["Asset Class"])
# name = [v.split("—")[0].strip() for v in asset_list]
# perc = [v.split("—")[1] for v in asset_list]

# df_asset_class_level_analysis = df_asset_class_level_analysis.drop(columns=["Asset Class"])
# df_asset_class_level_analysis.insert(0, "Assets", name)
# df_asset_class_level_analysis.insert(1, "percentage", perc)

# df_asset_class_level_analysis

In [130]:
column_mapping = {
    "investments": "Investments",
    "reference_rate": "Reference Rate and Spread",
    "interest_rate": "Interest Rate",	
    "industry": "Industry",
    "instrument": "Instrument",
    "investment_type": "Investment Type",
    "acquisition_date": "Acquisition Date",
    "maturity_date": "Maturity Date",
    "par_amount_units": "Par Amount / Units",
    "net_assets": "% of Net Assets",
    "cost": "Cost",
    "fair_value": "Fair Value",
}

desired_order = [
    "Investments",
    "Reference Rate and Spread",
    "Interest Rate",
    "Acquisition Date",
    "Maturity Date",
    "Par Amount / Units",
    "Cost",
    "Fair Value",
    "% of Net Assets",
    "Investment Type",
    "Instrument",
    "Industry",
]

df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Investments,Reference Rate and Spread,Interest Rate,Acquisition Date,Maturity Date,Par Amount / Units,Cost,Fair Value,% of Net Assets,Investment Type,Instrument,Industry
0,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.22%,7/25/2025,5/25/2029,41906,29763,8882,0.02,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
1,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,7/25/2025,5/25/2029,6096,4373,1292,0.00,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
2,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,7/25/2025,5/25/2029,37217,24850,6792,0.02,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
3,Atlas CC Acquisition Corp.,SOFR + 4.00 %,7.93%,7/25/2025,5/25/2029,18518,13604,18193,0.04,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
4,"Corfin Holdings, Inc.",SOFR + 5.25 %,9.02%,1/7/2021,12/27/2027,32050,32034,32050,0.07,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
...,...,...,...,...,...,...,...,...,...,...,...,...
1288,BCRED Verdelite JV LP - LP Interest,,NaN,10/21/2022,NaN,NaN,117706,97149,0.22,investments in joint ventures,equity and other,specialty retail
1289,Cash and Cash Equivalents,Cash and Cash Equivalents Cash and Cash Equiva...,Cash and Cash Equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,Cash and Cash Equivalents,<NA>,<NA>,NaN,cash and cash equivalents,equity and other,specialty retail
1290,Fidelity Investments Money Market Treasury Por...,,3.55%,NaN,NaN,NaN,6563,6563,0.01,cash and cash equivalents,equity and other,specialty retail
1291,State Street Institutional U.S. Government Mon...,,3.52%,NaN,NaN,NaN,10440,10440,0.02,cash and cash equivalents,equity and other,specialty retail


In [131]:
df_investment_formatted = df_investment_formatted[df_investment_formatted["Investments"] != "Cash and Cash Equivalents"]

### Dumping to GS

In [132]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [133]:
df_map = {
    "Summary": df_summary,
    "All Investments": df_investment_formatted,
    "Investment Level Analysis": df_investments_level_analysis,
    "Industry Level Analysis": df_industry_level_analysis,
    "Investment Type level Analysis": df_investmentsType_level_analysis
}

In [134]:
spreadsheet = client.open("BPCF-10Q-20260331")

In [135]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)
    
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

,a,b
0,Total Investment Count,1293
1,Total Cost,83006982
2,Total FV,81123110
3,P&L,-1883872


,Investments,Reference Rate and Spread,Interest Rate,Acquisition Date,Maturity Date,Par Amount / Units,Cost,Fair Value,% of Net Assets,Investment Type,Instrument,Industry
0,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.22%,7/25/2025,5/25/2029,41906,29763,8882,0.02,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
1,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,7/25/2025,5/25/2029,6096,4373,1292,0.00,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
2,Atlas CC Acquisition Corp.,SOFR + 4.25 %,8.18%,7/25/2025,5/25/2029,37217,24850,6792,0.02,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
3,Atlas CC Acquisition Corp.,SOFR + 4.00 %,7.93%,7/25/2025,5/25/2029,18518,13604,18193,0.04,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
4,"Corfin Holdings, Inc.",SOFR + 5.25 %,9.02%,1/7/2021,12/27/2027,32050,32034,32050,0.07,first lien debt - non-controlled/non-affiliated,first lien debt,aerospace & defense
...,...,...,...,...,...,...,...,...,...,...,...,...
1287,BCRED Emerald JV LP - LP Interest,,NaN,1/19/2022,NaN,NaN,1836000,1539677,3.41,investments in joint ventures,equity and other,specialty retail
1288,BCRED Verdelite JV LP - LP Interest,,NaN,10/21/2022,NaN,NaN,117706,97149,0.22,investments in joint ventures,equity and other,specialty retail
1290,Fidelity Investments Money Market Treasury Por...,,3.55%,NaN,NaN,NaN,6563,6563,0.01,cash and cash equivalents,equity and other,specialty retail
1291,State Street Institutional U.S. Government Mon...,,3.52%,NaN,NaN,NaN,10440,10440,0.02,cash and cash equivalents,equity and other,specialty retail


,Investments,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
259,"Dropbox, Inc.",2,1603086,1599228,-3858,-1929.0,-1358,-2500,0,0.0,0,0.0,2,1.0
420,"IRI Group Holdings, Inc.",1,1571509,1588160,16651,16651.0,16651,16651,1,1.0,0,0.0,0,0.0
81,BCRED Emerald JV LP - LP Interest,1,1836000,1539677,-296323,-296323.0,-296323,-296323,0,0.0,0,0.0,1,1.0
387,"Guidehouse, Inc.",1,1245071,1251898,6827,6827.0,6827,6827,1,1.0,0,0.0,0,0.0
433,"Inovalon Holdings, Inc.",2,1257670,1207576,-50094,-25047.0,-14949,-35145,0,0.0,0,0.0,2,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,"Material+ Holding Company, LLC - Class A Prefe...",1,0,0,0,0.0,0,0,0,0.0,1,1.0,0,0.0
147,"CFCo, LLC (Benefytt Technologies, Inc.) - Clas...",1,0,0,0,0.0,0,0,0,0.0,1,1.0,0,0.0
146,"CFCo, LLC (Benefytt Technologies, Inc.)",1,12571,0,-12571,-12571.0,-12571,-12571,0,0.0,0,0.0,1,1.0
126,"Box Co-Invest Blocker, LLC - (BP Alpha Holding...",1,390,0,-390,-390.0,-390,-390,0,0.0,0,0.0,1,1.0


,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
42,software,221,22328547,21560513,-768034,-3475.27,44246,-329098,104,0.47,4,0.02,113,0.51
39,professional services,106,8956217,8937116,-19101,-180.2,13127,-57075,64,0.6,5,0.05,37,0.35
7,commercial services & supplies,89,6585998,6597267,11269,126.62,4507,-5967,59,0.66,3,0.03,27,0.3
23,health care providers & services,94,6910543,6447857,-462686,-4922.19,9106,-109033,28,0.3,7,0.07,59,0.63
28,insurance,75,4872323,4860613,-11710,-156.13,12047,-41038,49,0.65,6,0.08,20,0.27
24,health care technology,47,4426905,4446500,19595,416.91,30653,-10217,19,0.4,1,0.02,27,0.57
30,it services,43,3868043,3768147,-99896,-2323.16,17700,-35145,18,0.42,0,0.0,25,0.58
19,financial services,164,2496711,2428757,-67954,-414.35,9816,-10958,25,0.15,1,0.01,138,0.84
43,specialty retail,13,2686800,2364850,-321950,-26829.17,241,-296323,3,0.23,3,0.23,6,0.46
11,diversified consumer services,39,2316913,2286436,-30477,-781.46,8756,-24758,15,0.38,0,0.0,24,0.62


,Investment Type,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
6,first lien debt - non-controlled/non-affiliated,972,73626433,72447160,-1179273,-1213.24,44246,-329098,509,0.52,19,0.02,444,0.46
8,second lien debt - non-controlled/non-affiliated,46,3283897,3051931,-231966,-5042.74,2907,-41046,11,0.24,1,0.02,34,0.74
7,investments in joint ventures,2,1953706,1636826,-316880,-158440.0,-20557,-296323,0,0.0,0,0.0,2,1.0
1,equity and other - controlled/affiliated (excl...,12,1210988,1243135,32147,2922.45,12834,0,4,0.33,7,0.58,0,0.0
3,equity and other - non-controlled/non-affiliated,72,689503,749088,59585,827.57,30653,-45455,41,0.57,9,0.12,22,0.31
0,cash and cash equivalents,4,653729,653729,0,0.0,0,0,0,0.0,3,0.75,0,0.0
4,first lien debt - controlled/affiliated,10,697770,551473,-146297,-14629.7,64,-57075,1,0.1,1,0.1,8,0.8
9,structured finance obligations - debt instrume...,98,437449,412348,-25101,-256.13,53,-1830,3,0.03,0,0.0,95,0.97
10,structured finance obligations - equity instru...,47,329028,256137,-72891,-1550.87,1163,-10958,8,0.17,1,0.02,38,0.81
11,unsecured debt - non-controlled/non-affiliated,24,82894,80108,-2786,-116.08,72,-2728,4,0.17,0,0.0,20,0.83


In [39]:
for i in df_investment['asset_class'].unique(): print(i)

aerospace & defense
air freight & logistics
airlines (passenger airlines)
biotechnology
building products
capital markets
chemicals
commercial services & supplies
construction & engineering
containers & packaging
distributors
diversified consumer services
diversified reits
diversified telecommunication services
electric utilities
electrical equipment
electronic equipment, instruments & components
energy equipment & services
entertainment
financial services
food products
ground transportation
health care equipment & supplies
health care providers & services
health care technology
hotels, restaurants & leisure
household durables
industrial conglomerates
insurance
interactive media & services
it services
life sciences tools & services
machinery
marine transportation
media
metals & mining
oil, gas & consumable fuels
paper & forest products
pharmaceuticals
professional services
real estate management & development
semiconductors & semiconductor equipment
software
specialty retail
technology